# Visualizing ghost molecules

<video controls src="./assets/ghost_molecules.webm">


Load the nanotube + methane example recording as an MDAnalsis universe:

In [1]:
from nanover.mdanalysis import universe_from_recording

universe = universe_from_recording("../recordings/nanotube-example-recording.nanover.zip")

Set up the server with playback of the universe:

In [2]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: trajectory seeker")

Import the jupyter utilities for drawing + interaction:

In [3]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Define a FrameListener to constantly update the ghost visuals as the nanotube moves during playback:

In [4]:
from nanover.utilities.transforms import StructureAlignment
from nanover.jupyter.alignment_transform import AlignmentTransform

NANOTUBE_ATOMS = range(0, 60)
alignment = StructureAlignment.from_atom_group(universe.atoms[NANOTUBE_ATOMS])

alignment_update = AlignmentTransform.from_runner(imd_runner)
alignment_update.config(key="nanotube", alignment=alignment)
alignment_update.start()

Define a function to extract ghost molecules from particular trajectory frames and translate them into a common reference frame:

In [5]:
import MDAnalysis as mda


def add_ghost(frame_index, atoms):
    _ = universe.trajectory[frame_index]

    # find relative transform between nanotube initial pose and nanotube pose in this frame
    transform_i = alignment.calculate_transform_to_universe(universe)

    # extract ghost molecule and transform position in this frame to same position relative to nanotube in first frame
    ghost_universe = mda.Merge(universe.atoms[atoms])
    ghost_positions = transform_i.points_local_to_parent(ghost_universe.atoms.positions / 10)
    ghost_bond_pairs = ghost_universe.bonds.indices

    # add transparent spheres and lines to scene at positions relative to nanotube in first frame:
    for i, position in enumerate(ghost_positions):
        utilities.objects.update_shape(f"ghost.{frame_index}.{i}", position=position, size=0.1, color=[1.0, 1.0, 1.0, 0.25], parent="nanotube")
    for i, (a, b) in enumerate(ghost_bond_pairs):
        utilities.objects.update_line(f"ghost.{frame_index}.{i}", positions=ghost_positions[[a, b]], size=0.075, color=[1.0, 1.0, 1.0, 0.5], parent="nanotube")

Add ghosts for methane atoms at particular frames:

In [6]:
METHANE_ATOMS = list(range(60, 65))

for frame_index in [500, 610, 775, len(universe.trajectory)-1]:
    add_ghost(frame_index, METHANE_ATOMS)

Start simulation running:

In [7]:
imd_runner.load(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt
